In [0]:
from pyspark.sql.functions import col, expr, sum, count, desc

data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]

columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

df = spark.createDataFrame(data, columns)

df = df.withColumn(
    "total_amount",
    col("consultation_fee") + (col("tests_count") * 500)
)

df = df.withColumn(
    "patient_type",
    expr("CASE WHEN total_amount >= 6000 THEN 'High Value' ELSE 'Normal' END")
)

print("Complete DataFrame with Derived Columns:")
df.show()

high_value_df = df.filter(col("total_amount") >= 6000)

print("High Value Patients:")
high_value_df.show()

dept_summary = df.groupBy("department") \
    .agg(
        count("*").alias("patient_count"),
        sum("total_amount").alias("department_revenue")
    )

print("Department Wise Summary:")
dept_summary.show()

dept_summary_sorted = dept_summary.orderBy(desc("department_revenue"))

print("Department Wise Summary Sorted by Revenue:")
dept_summary_sorted.show()

Complete DataFrame with Derived Columns:
+--------+------------+---------+-----------+----------------+-----------+------------+------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_amount|patient_type|
+--------+------------+---------+-----------+----------------+-----------+------------+------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|        5500|      Normal|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|        4000|      Normal|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|        2000|      Normal|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|        6000|  High Value|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|        7500|  High Value|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|        4500|      Normal|
|     107| Karan Patel|Ahmedabad| Cardiology|     

In [0]:
df.createOrReplaceTempView("hospital_visits")

In [0]:
%sql
SELECT *
FROM hospital_visits
WHERE department = 'Cardiology';


visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Normal
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High Value
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Normal


In [0]:
%sql
SELECT 
    city,
    SUM(total_amount) AS city_revenue
FROM hospital_visits
GROUP BY city
ORDER BY city_revenue DESC;


city,city_revenue
Bangalore,8500
Chennai,7500
Hyderabad,5500
Ahmedabad,5500
Kolkata,4500
Delhi,4000
Mumbai,2000


In [0]:
%sql
SELECT 
    patient_name,
    city,
    department,
    total_amount
FROM hospital_visits
ORDER BY total_amount DESC
LIMIT 5;


patient_name,city,department,total_amount
Vikram Singh,Chennai,Neurology,7500
Priya Nair,Bangalore,Cardiology,6000
Karan Patel,Ahmedabad,Cardiology,5500
Arjun Reddy,Hyderabad,Cardiology,5500
Ananya Das,Kolkata,Orthopedics,4500


In [0]:
%sql

SELECT 
    department,
    COUNT(*) AS patient_count
FROM hospital_visits
GROUP BY department
ORDER BY patient_count DESC;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("hospital_delta_table")

display(spark.sql("SELECT * FROM hospital_delta_table"))

visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Normal
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,Normal
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,Normal
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High Value
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High Value
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Normal
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Normal
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Normal


In [0]:
%sql

INSERT INTO hospital_delta_table
VALUES 
(109, 'Divya Menon', 'Pune', 'Cardiology', 5000, 2, 6000, 'High Value'),
(110, 'Nikhil Rao', 'Coimbatore', 'Neurology', 7000, 2, 8000, 'High Value');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql

UPDATE hospital_delta_table
SET 
    consultation_fee = 8000,
    total_amount = 8500,
    patient_type = 'High Value'
WHERE visit_id = 103;

num_affected_rows
1


In [0]:
%sql

DELETE FROM hospital_delta_table
WHERE visit_id = 108;

num_affected_rows
1


In [0]:
updates_data = [
(102,"Sneha Kapoor","Delhi","Orthopedics",3500,2,4500,"Normal"),
(111,"Riya Thomas","Madurai","Dermatology",2000,3,3500,"Normal")
]

updates_columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count",
"total_amount",
"patient_type"
]

updates_df = spark.createDataFrame(updates_data, updates_columns)

updates_df.createOrReplaceTempView("hospital_updates")

In [0]:
%sql

MERGE INTO hospital_delta_table AS target
USING hospital_updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql

MERGE INTO hospital_delta_table AS target
USING hospital_updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
%sql

SELECT *
FROM hospital_delta_table
ORDER BY visit_id;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Normal
102,Sneha Kapoor,Delhi,Orthopedics,3500,2,4500,Normal
103,Rahul Sharma,Mumbai,Dermatology,8000,1,8500,High Value
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High Value
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High Value
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Normal
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Normal
109,Divya Menon,Pune,Cardiology,5000,2,6000,High Value
110,Nikhil Rao,Coimbatore,Neurology,7000,2,8000,High Value
111,Riya Thomas,Madurai,Dermatology,2000,3,3500,Normal


In [0]:
%sql

DESCRIBE HISTORY hospital_delta_table;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-05-04T04:39:47.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4335410207079590),189febbb-bfb3-4bbd-b262-5c3d3b25e3de,0504-041524-mw9m4gqp-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7506, p25FileSize -> 2862, numDeletionVectorsRemoved -> 1, minFileSize -> 2862, numAddedFiles -> 1, maxFileSize -> 2862, p75FileSize -> 2862, p50FileSize -> 2862, numAddedBytes -> 2862)",null,Databricks-Runtime/18.1.x-photon-scala2.13
7,2026-05-04T04:39:45.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#15757L = visit_id#15101L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4335410207079590),189febbb-bfb3-4bbd-b262-5c3d3b25e3de,0504-041524-mw9m4gqp-v2n,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4644, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3527, materializeSourceTimeMs -> 81, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1509, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1912)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T04:39:19.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4335410207079590),d2723115-c9f0-41ae-b833-b3a2e446f1e5,0504-041524-mw9m4gqp-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7541, p25FileSize -> 2862, numDeletionVectorsRemoved -> 1, minFileSize -> 2862, numAddedFiles -> 1, maxFileSize -> 2862, p75FileSize -> 2862, p50FileSize -> 2862, numAddedBytes -> 2862)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-04T04:39:18.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#15159L = visit_id#15101L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4335410207079590),d2723115-c9f0-41ae-b833-b3a2e446f1e5,0504-041524-mw9m4gqp-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4644, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4193, materializeSourceTimeMs -> 487, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 1624, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2028)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T04:38:14.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(visit_id#14878L = 108)""])",null,List(4335410207079590),2288719b-fac1-4d53-9597-757a99dabe05,0504-041524-mw9m4gqp-v2n,3,WriteSerializable,false

In [0]:
%sql

SELECT *
FROM hospital_delta_table VERSION AS OF 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Normal
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,Normal
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,Normal
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High Value
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High Value
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Normal
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Normal
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Normal


In [0]:
%sql

SELECT *
FROM hospital_delta_table VERSION AS OF 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Normal
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,Normal
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,Normal
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High Value
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High Value
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Normal
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Normal
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Normal


In [0]:
%sql

SELECT *
FROM hospital_delta_table
ORDER BY visit_id;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Normal
102,Sneha Kapoor,Delhi,Orthopedics,3500,2,4500,Normal
103,Rahul Sharma,Mumbai,Dermatology,8000,1,8500,High Value
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High Value
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High Value
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Normal
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Normal
109,Divya Menon,Pune,Cardiology,5000,2,6000,High Value
110,Nikhil Rao,Coimbatore,Neurology,7000,2,8000,High Value
111,Riya Thomas,Madurai,Dermatology,2000,3,3500,Normal


In [0]:
%sql

VACUUM hospital_delta_table DRY RUN;

path


In [0]:
parquet_table = "hospital_parquet_table"

df.write.mode("overwrite") \
    .format("parquet") \
    .saveAsTable(parquet_table)

print("Parquet table created successfully")

In [0]:
%sql

SELECT *
FROM hospital_parquet_table;

In [0]:
%sql

CREATE OR REPLACE TABLE hospital_converted_delta
USING DELTA
AS
SELECT *
FROM hospital_parquet_table;

In [0]:
%sql

SELECT *
FROM hospital_converted_delta;

In [0]:
%sql

DESCRIBE DETAIL hospital_converted_delta;

In [0]:
from pyspark.sql.functions import col, expr

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("hospital_incremental_target")

print("Target Delta table created successfully")

Target Delta table created successfully


In [0]:
daily_updates_data = [
    (102, "Sneha Kapoor", "Delhi", "Orthopedics", 3500, 3),
    (109, "Divya Menon", "Pune", "Cardiology", 5000, 2),
    (110, "Nikhil Rao", "Coimbatore", "Neurology", 7000, 1)
]

daily_columns = [
    "visit_id",
    "patient_name",
    "city",
    "department",
    "consultation_fee",
    "tests_count"
]

daily_df = spark.createDataFrame(daily_updates_data, daily_columns)

daily_df = daily_df.withColumn(
    "total_amount",
    col("consultation_fee") + (col("tests_count") * 500)
)

daily_df = daily_df.withColumn(
    "patient_type",
    expr("CASE WHEN total_amount >= 6000 THEN 'High Value' ELSE 'Normal' END")
)

daily_df.createOrReplaceTempView("daily_updates")

display(daily_df)

visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
102,Sneha Kapoor,Delhi,Orthopedics,3500,3,5000,Normal
109,Divya Menon,Pune,Cardiology,5000,2,6000,High Value
110,Nikhil Rao,Coimbatore,Neurology,7000,1,7500,High Value


In [0]:
%sql

MERGE INTO hospital_incremental_target AS target
USING daily_updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,0,2


In [0]:
%sql

SELECT *
FROM hospital_incremental_target
ORDER BY visit_id;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_amount,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Normal
102,Sneha Kapoor,Delhi,Orthopedics,3500,3,5000,Normal
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,Normal
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High Value
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High Value
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Normal
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Normal
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Normal
109,Divya Menon,Pune,Cardiology,5000,2,6000,High Value
110,Nikhil Rao,Coimbatore,Neurology,7000,1,7500,High Value


In [0]:
import dlt
from pyspark.sql.functions import col, expr, sum, count

# 1. Bronze table - raw inline data
@dlt.table(
    name="bronze_hospital_visits",
    comment="Raw hospital visit data"
)
def bronze_hospital_visits():
    data = [
        (101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
        (102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
        (103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
        (104,"Priya Nair","Bangalore","Cardiology",5000,2),
        (105,"Vikram Singh","Chennai","Neurology",7000,1),
        (106,"Ananya Das","Kolkata","Orthopedics",3000,3),
        (107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
        (108,"Meera Iyer","Bangalore","Dermatology",1500,2)
    ]

    columns = [
        "visit_id",
        "patient_name",
        "city",
        "department",
        "consultation_fee",
        "tests_count"
    ]

    return spark.createDataFrame(data, columns)

@dlt.table(
    name="silver_hospital_visits",
    comment="Cleaned hospital visit data with derived columns"
)
def silver_hospital_visits():
    return dlt.read("bronze_hospital_visits") \
        .withColumn(
            "total_amount",
            col("consultation_fee") + (col("tests_count") * 500)
        ) \
        .withColumn(
            "patient_type",
            expr("CASE WHEN total_amount >= 6000 THEN 'High Value' ELSE 'Normal' END")
        )

@dlt.table(
    name="gold_department_revenue",
    comment="Department-wise patient count and revenue"
)
def gold_department_revenue():
    return dlt.read("silver_hospital_visits") \
        .groupBy("department") \
        .agg(
            count("*").alias("patient_count"),
            sum("total_amount").alias("department_revenue")
        )

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS hospital_catalog;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS hospital_catalog.assessment_schema;

In [0]:
%sql

CREATE OR REPLACE TABLE hospital_catalog.assessment_schema.hospital_visits
USING DELTA
AS
SELECT *
FROM hospital_delta_table;

In [0]:
%sql

SELECT *
FROM hospital_catalog.assessment_schema.hospital_visits;

In [0]:
%sql

SHOW CATALOGS;

SHOW SCHEMAS IN hospital_catalog;

SHOW TABLES IN hospital_catalog.assessment_schema;

In [0]:
%sql

CREATE OR REPLACE TABLE hospital_catalog.assessment_schema.department_revenue_summary
USING DELTA
AS
SELECT 
    department,
    COUNT(*) AS patient_count,
    SUM(total_amount) AS department_revenue
FROM hospital_catalog.assessment_schema.hospital_visits
GROUP BY department;

In [0]:
%sql

SELECT *
FROM hospital_catalog.assessment_schema.department_revenue_summary
ORDER BY department_revenue DESC;

In [0]:
%sql

GRANT SELECT 
ON TABLE hospital_catalog.assessment_schema.hospital_visits 
TO `account users`;

In [0]:
%sql

SELECT 
    event_time,
    user_identity.email,
    action_name,
    service_name
FROM system.access.audit
ORDER BY event_time DESC
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE TABLE hospital_catalog.assessment_schema.final_hospital_analytics
USING DELTA
AS
SELECT 
    department,
    city,
    COUNT(*) AS patient_count,
    SUM(total_amount) AS total_revenue,
    AVG(total_amount) AS average_revenue
FROM hospital_catalog.assessment_schema.hospital_visits
GROUP BY department, city;

In [0]:
%sql
SELECT *
FROM hospital_catalog.assessment_schema.final_hospital_analytics
ORDER BY total_revenue DESC;